# 1.8 猜錯要付出多少代價？

# 第 1 章：從猜下一個字開始

前置：讀過入門指南的 Python 與 tensor 部分即可。這一章只用幾個字；不要先開完整 Transformer。每段 code 都可單獨執行。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：把分數換成比例：像分一塊餅**

一起加同常數不改比例，分數差才重要。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/softmax.svg")))

**先想一想：**正確機率從 0.5 升到 0.9，loss 應該上升嗎？

答案越不被相信，代價越大。cross-entropy 對一個確定答案就是正確答案機率的負對數；它不是「猜錯一次就扣一分」。

**把直覺寫成數學：**$L=-\ln p_y$；當 $p_y\to0$，loss 變大。使用自然對數時單位是 nats。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
p = torch.tensor([0.1, 0.5, 0.9])
print("正確機率", p, "loss", -p.log())
logits = torch.tensor([[0.0, 2.0, 0.0]])
assert torch.allclose(F.cross_entropy(logits, torch.tensor([1])), -logits.softmax(-1)[0, 1].log())

**如何讀結果：**同一批資料才適合直接比較；不同 tokenizer 的 token 平均值先不要混比。

**只改一件事：**只把正確類別換成分數最低者，觀察代價。
